In [ ]:
 import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from google.colab import drive
from torchsummary import summary

In [ ]:
# 1. Mount Google Drive
drive.mount('/content/gdrive')

# 2. Load labeled images from folders
# data_dir = '/content/gdrive/MyDrive/Data1/antsbeesdataset'
data_dir = '/content/gdrive/MyDrive/Data2023/ballgame'

In [ ]:
cd /content/gdrive/MyDrive/Data2023/ballgame

In [ ]:
ls

In [ ]:
# 3. Pre-process the data and create data loaders
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'valid': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

In [ ]:
image_datasets = {x: datasets.ImageFolder(data_dir + '/' + x, data_transforms[x]) for x in ['train', 'valid']}
dataloaders = {x: DataLoader(image_datasets[x], batch_size=16, shuffle=True, num_workers=4) for x in ['train', 'valid']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'valid']}
class_names = image_datasets['train'].classes
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
# 4. Set up the AlexNet architecture
alexnet = models.alexnet(pretrained=True)

In [ ]:
alexnet

In [ ]:

num_ftrs = alexnet.classifier[6].in_features
alexnet.classifier[6] = nn.Linear(num_ftrs, len(class_names)) # Change last layer
alexnet = alexnet.to(device) # Put on GPU

In [ ]:
alexnet

In [ ]:
summary(alexnet, (3, 224, 224))

In [ ]:
class_names

In [ ]:
dataset_sizes

In [ ]:
for inputs, labels in dataloaders["train"]:
        inputs = inputs.to(device)
        labels = labels.to(device)

In [ ]:
inputs.shape

In [ ]:
labels.shape

In [ ]:
labels

In [ ]:
outputs = alexnet(inputs)

In [ ]:
outputs.shape

In [ ]:
outputs

In [ ]:
torch.max(outputs,1)[0]

In [ ]:
torch.max(outputs,1)[1]

In [ ]:
preds = torch.max(outputs, 1)[1]

In [ ]:
preds

In [ ]:
labels

In [ ]:
preds == labels

In [ ]:
labels.shape[0]

In [ ]:
torch.sum(preds == labels)/labels.shape[0]

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(alexnet.parameters(), lr=0.001, momentum=0.9)

# Train the AlexNet model
alexnet.train()

num_epochs = 10

acc_best = 0
parameters_best = 0

for epoch in range(num_epochs):
    print(epoch, " of ", num_epochs - 1)
    print('-' * 10)

    running_corrects = 0

    for inputs, labels in dataloaders["train"]:
        inputs = inputs.to(device)
        labels = labels.to(device)


        outputs = alexnet(inputs)
        preds = torch.max(outputs, 1)[1]


        optimizer.zero_grad()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_corrects += torch.sum(preds == labels.data)

    print('Train Acc: {:.4f}'.format(running_corrects / dataset_sizes["train"]))


    # Evaluate the AlexNet model on Validation Data
    alexnet.eval()

    running_corrects = 0

    for inputs, labels in dataloaders["valid"]:
        inputs = inputs.to(device)
        labels = labels.to(device)


        outputs = alexnet(inputs)
        preds = torch.max(outputs, 1)[1]


        running_corrects += torch.sum(preds == labels.data)

    acc_valid = running_corrects / dataset_sizes["valid"]

    if acc_valid > acc_best:
        acc_best= acc_valid
        parameters_best = alexnet.parameters()


    print('Valid Acc: {:.4f}'.format(acc_valid))
    if acc_valid > 0.99:
        print("Done!")
        break

alexnet.parameters = parameters_best
print('Training complete')

In [ ]:
pwd

In [ ]:
model_path = '/content/gdrive/MyDrive/Data2023/ballgame/alexnet_classification_ballgame.pth'

In [ ]:
# Save the trained model
torch.save(alexnet.state_dict(), model_path)

In [ ]:
ls

In [ ]:
import requests
from PIL import Image
from io import BytesIO

# 1. Load the trained AlexNet model
def load_model(model_path):
    model = models.alexnet()
    num_ftrs = model.classifier[6].in_features
    model.classifier[6] = nn.Linear(num_ftrs, len(class_names))
    model.load_state_dict(torch.load(model_path))
    model.eval()
    return model.to(device)

# 2. Define a function to load an image from a URL and preprocess it
def preprocess_image(url, transform):
    response = requests.get(url)
    img = Image.open(BytesIO(response.content)).convert("RGB")
    img_tensor = transform(img)
    return img_tensor.unsqueeze(0).to(device)

# 3. Perform inference using the loaded model
def predict_image_url(url, model):
    img_tensor = preprocess_image(url, data_transforms['valid'])
    output = model(img_tensor)
    pred = torch.max(output, 1)[1]
    return class_names[pred]

In [ ]:
trained_model = load_model(model_path)

In [ ]:
image_url = 'https://cdnph.upi.com/ph/st/th/6151680015900/2023/upi_com/d27c0d26c779599eb00767c375dbd848/v1.2/Mens-college-basketball-FAU-San-Diego-St-to-tip-off-unexpected-Final-Four.jpg'

In [ ]:
prediction = predict_image_url(image_url, trained_model)

In [ ]:
print("The predicted class for the input image is:", prediction)

In [ ]:
image = preprocess_image(image_url,data_transforms['valid'])

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.imshow(image.cpu().numpy().transpose(0,2,3,1)[0])

In [ ]:
image_attack = image.clone().to(device)
image_attack.requires_grad = True

In [ ]:
def image_normalize(x):
    return (x - x.min())/(x.max() - x.min())

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD([image_attack], lr=0.001, momentum=0.9)
target_label = torch.tensor([1]).to(device)
# Train the AlexNet model
alexnet.eval()

num_epochs = 200

for epoch in range(num_epochs):
    print(epoch, " of ", num_epochs - 1)
    print('-' * 10)

    running_corrects = 0

    optimizer.zero_grad()

    outputs = alexnet(image_attack)
    preds = torch.max(outputs, 1)[1]

    loss = criterion(outputs, target_label)
    loss.backward()
    optimizer.step()

    print(loss.item(),preds.item())
    if epoch % 10 == 0:
        plt.imshow(image_normalize(image_attack.detach().cpu().numpy().transpose(0,2,3,1)[0]))
        plt.show()
        plt.imshow(image_normalize(image_attack.grad.detach().cpu().numpy().transpose(0,2,3,1)[0]))
        plt.show()



print('Training complete')

In [ ]:
plt.imshow(image_normalize(image.cpu().numpy().transpose(0,2,3,1)[0]))
plt.show()
plt.imshow(image_normalize(image_attack.detach().cpu().numpy().transpose(0,2,3,1)[0]))
plt.show()
plt.imshow(image_normalize((image_attack.detach().cpu().numpy().transpose(0,2,3,1)[0] - image.cpu().numpy().transpose(0,2,3,1)[0])))
plt.show()

In [ ]:
image_new_url = "https://cdn.pixabay.com/photo/2016/07/02/04/27/basketball-1492401_640.jpg"

In [ ]:
prediction = predict_image_url(image_new_url, trained_model)

In [ ]:
print("The predicted class for the input image is:", prediction)

In [ ]:
image_delta = (image_attack.detach().cpu().numpy().transpose(0,2,3,1)[0] - image.cpu().numpy().transpose(0,2,3,1)[0])

In [ ]:
image_new = preprocess_image(image_new_url,data_transforms['valid'])

In [ ]:
plt.imshow(image_normalize(image_new.cpu().numpy().transpose(0,2,3,1)[0]))
plt.show()
plt.imshow(image_normalize(image_new.detach().cpu().numpy().transpose(0,2,3,1)[0]) + image_delta)
plt.show()

In [ ]:
alexnet((image_attack.detach() - image) + image_new)

In [ ]:
def block_section(img, x, y, w, h):
    img_masked = img.clone()
    img_masked[..., x:x+w, y:y+h] = 0
    return img_masked

In [ ]:
import numpy as np

In [ ]:
mask = np.random.randint(0,224-50,size=(2))
img_masked = block_section(image, *mask, 50, 50)

In [ ]:
mask

In [ ]:
plt.imshow(image_normalize(img_masked.cpu().numpy().transpose(0,2,3,1)[0]))
plt.show()

In [ ]:
alexnet.eval()

num_epochs = 100

for epoch in range(num_epochs):

    mask = np.random.randint(0,224-200,size=(2))
    img_masked = block_section(image, *mask, 200, 200)
    outputs = alexnet(img_masked)
    preds = torch.max(outputs, 1)[1]

    print(preds.item())
    if preds.item() == 1:
        plt.imshow(image_normalize(img_masked.detach().cpu().numpy().transpose(0,2,3,1)[0]))
        plt.show()

In [ ]:
img_masked = block_section(image, 125, 125, 100, 100)
outputs = alexnet(img_masked)
preds = torch.max(outputs, 1)[1]

print(preds.item())
plt.imshow(image_normalize(img_masked.detach().cpu().numpy().transpose(0,2,3,1)[0]))
plt.show()

In [ ]:
img_football_url = "https://upload.wikimedia.org/wikipedia/commons/2/23/Wilson_American_football.jpg"

In [ ]:
image_football = preprocess_image(img_football_url,data_transforms['valid'])